# Baseline Benchmarks Notebook

Run the cells top-to-bottom in order. This notebook starts from cloning the repository, then installs requirements, then reproduces the one-example GovReport walkthrough, and finally runs all-SCROLLS latency + attention-map benchmarks.


In [ ]:
%%bash
set -e
if [ ! -d MLSFINAL ]; then
  git clone --branch pranav https://github.com/thetechdude124/MLSFINAL.git
fi


In [ ]:
%cd MLSFINAL
!git checkout pranav


In [ ]:
!pip install -q -r requirements.txt


This file contains all the necessary information for running baseline benchmarks of small transformer
models with full attention on long-context tasks. The initial baseline will use pretrained weights from 
GPT-2 and Pythia-160M, running on the PG-19 (Project Gutenberg books) and SCROLLS (a set of 
long-context reasoning benchmarks) datasets. 


In [ ]:
from dotenv import load_dotenv
import os
import time

import matplotlib.pyplot as plt
import pandas as pd
import torch
from datasets import get_dataset_config_names, load_dataset
from evaluate import load
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, GPT2LMHeadModel

load_dotenv()
print("Running baselines.py")


Step 1: Download the data from PG-19 and SCROLLS. Store it in the 'data' directory. 

Note: For now, we only load the gov_report task of SCROLLS (since I don't have enough local disk space)


In [ ]:
cache_dir = "./data"
# pg19 = load_dataset("pg19", cache_dir=cache_dir)
configs = [
    c for c in get_dataset_config_names("tau/scrolls")
    if c != "narrative_qa"
]
print(configs)
scrolls = {}

for name in tqdm(configs, desc="Loading SCROLLS"):
    print(name)
    scrolls[name] = load_dataset(
        "tau/scrolls",
        name,
        cache_dir=cache_dir,
        trust_remote_code=True
    )
scrolls = load_dataset("tau/scrolls", "gov_report", cache_dir=cache_dir, trust_remote_code=True)


Step 2:
Load the pretrained model weights for Pythia-160M and GPT-2


In [ ]:
gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")
gpt2_model = AutoModelForCausalLM.from_pretrained("gpt2")

pythia_tokenizer = AutoTokenizer.from_pretrained("EleutherAI/pythia-160m")
pythia_model = AutoModelForCausalLM.from_pretrained("EleutherAI/pythia-160m")

# print(pg19)
print(scrolls)


Step 3: Benchmark how well each model predicts the data and how expensive it is to do so. 
+ Visualize additional metrics such as attention maps 

First, we benchmark Pythia-160M, then GPT-2. 


In [ ]:
model_name = "EleutherAI/pythia-160m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()


The gov_report task is to take in a government agency document and summarize it. 
This is presented as a sequence-to-sequence task, where the model takes in [DOCUMENT] + [SEP] + [SUMMARY]
and predicts the conditional probability of each token in the summary given all previous summary 
tokens and document tokens. 

The original SCROLLS paper aggregates ROUGE-1 (unigram overap) F1, ROUGE-2 (bigram overlap) F1 and 
ROUGE-L (longest overlapping subsequence) F1 to produce a single final ROUGE score. 


F1 = 2 * (precision * recall) / (precision + recall)
Calculating the harmonic mean forces both precision and recall to be high. 

We calculate the ROUGE score of Pythia-160M on one item in the GovReport validation dataset


In [ ]:
example = scrolls["validation"][0] 
inputs = tokenizer(example["input"], return_tensors="pt", truncation=True, max_length=1024)
output_ids = model.generate(**inputs, max_new_tokens=200)
prediction = tokenizer.decode(output_ids[0], skip_special_tokens=True)
reference = example["output"]

print("Computing ROUGE scores: ")
rouge = load("rouge")
scores = rouge.compute(
    predictions=[prediction],
    references=[reference]
)


Step 3: 
Now we compute the latency of Pythia-160M on one example from the SCROLLS/GovReport dataset
run k trials, take the average


In [ ]:
k = 5
latencies = []
tokens_generated = None

# warmup
_ = model.generate(**inputs, max_new_tokens=200)

for _ in range(k):
    start = time.time()
    out = model.generate(**inputs, max_new_tokens=200)
    end = time.time()

    latencies.append(end - start)

    if tokens_generated is None:
        tokens_generated = out.shape[1] - inputs["input_ids"].shape[1]

avg_latency = sum(latencies) / k
tokens_per_sec = tokens_generated / avg_latency

print("Avg Latency:", avg_latency)
print("Tokens/sec:", tokens_per_sec)

print(scores)


Step 4:
Next, we compute the attention map for that one such example


In [ ]:
model_name = "EleutherAI/pythia-160m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    attn_implementation="eager"
)
model.eval()

example = scrolls["validation"][0]

viz_inputs = tokenizer(
    example["input"],
    return_tensors="pt",
    truncation=True,
    max_length=1024,   # keep small enough to visualize clearly
)

with torch.no_grad():
    outputs = model(
        **viz_inputs,
        output_attentions=True
    )

# attentions is a tuple: one tensor per layer
# each tensor has shape [batch_size, num_heads, seq_len, seq_len]
layer_idx = 0
head_idx = 0

attn = outputs.attentions[layer_idx][0, head_idx].cpu()
tokens = tokenizer.convert_ids_to_tokens(viz_inputs["input_ids"][0])

plt.figure(figsize=(10, 8))
plt.imshow(attn, aspect="auto")
plt.colorbar()
plt.title(f"Attention Map | Layer {layer_idx} | Head {head_idx}")
plt.xlabel("Key position")
plt.ylabel("Query position")
plt.xticks(range(len(tokens)), tokens, rotation=90, fontsize=6)
plt.yticks(range(len(tokens)), tokens, fontsize=6)
plt.tight_layout()
plt.show()


The issue with our current algorithm is that we compute the top k attention scores per window — this means that
only tokens which are important in that window have a possibility of being remembered, even when they might 
be important globally. So instead of purely relying on our local sliding window attention scores, we should 
have a "global attention" metric that is trained to recognize globally important tokens even when they may not 
be important in the current window. The examples of this can come from computing full attention, identifying such 
tokens, and having some model analyze it for patterns. 


## Additional Notebook Cells: All SCROLLS Tasks

These cells are notebook additions so you can run the same style of measurements across all SCROLLS tasks. They keep the same seq2seq assumption and use one example per task.


In [ ]:
all_scrolls_configs = get_dataset_config_names("tau/scrolls")
print(all_scrolls_configs)

scrolls_all = {}
for name in tqdm(all_scrolls_configs, desc="Loading all SCROLLS tasks"):
    scrolls_all[name] = load_dataset(
        "tau/scrolls",
        name,
        cache_dir=cache_dir,
        trust_remote_code=True
    )


def get_first_seq2seq_example(dataset_dict):
    for split_name in ["validation", "train", "test"]:
        if split_name in dataset_dict and len(dataset_dict[split_name]) > 0:
            return dataset_dict[split_name][0], split_name
    raise ValueError("No non-empty split found for this task.")


In [ ]:
k_all = 3
all_latency_rows = []

benchmark_model_name = "EleutherAI/pythia-160m"
benchmark_tokenizer = AutoTokenizer.from_pretrained(benchmark_model_name)
benchmark_model = AutoModelForCausalLM.from_pretrained(benchmark_model_name)
benchmark_model.eval()

for task_name, task_dataset in tqdm(scrolls_all.items(), desc="Benchmarking latency for all SCROLLS tasks"):
    task_example, task_split = get_first_seq2seq_example(task_dataset)

    task_inputs = benchmark_tokenizer(
        task_example["input"],
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    _ = benchmark_model.generate(**task_inputs, max_new_tokens=200)

    task_latencies = []
    task_tokens_generated = None

    for _ in range(k_all):
        start = time.time()
        task_out = benchmark_model.generate(**task_inputs, max_new_tokens=200)
        end = time.time()

        task_latencies.append(end - start)

        if task_tokens_generated is None:
            task_tokens_generated = task_out.shape[1] - task_inputs["input_ids"].shape[1]

    task_avg_latency = sum(task_latencies) / k_all
    task_tokens_per_sec = task_tokens_generated / task_avg_latency

    all_latency_rows.append(
        {
            "task": task_name,
            "split_used": task_split,
            "avg_latency_sec": task_avg_latency,
            "tokens_per_sec": task_tokens_per_sec
        }
    )

latency_df = pd.DataFrame(all_latency_rows).sort_values("avg_latency_sec")
latency_df


In [ ]:
attention_model_name = "EleutherAI/pythia-160m"
attention_tokenizer = AutoTokenizer.from_pretrained(attention_model_name)
attention_model = AutoModelForCausalLM.from_pretrained(
    attention_model_name,
    attn_implementation="eager"
)
attention_model.eval()

layer_idx = 0
head_idx = 0
max_viz_length = 256

task_names = list(scrolls_all.keys())
n_tasks = len(task_names)
ncols = 3
nrows = (n_tasks + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = axes.flatten()

for i, task_name in enumerate(tqdm(task_names, desc="Computing one attention map per SCROLLS task")):
    task_dataset = scrolls_all[task_name]
    task_example, task_split = get_first_seq2seq_example(task_dataset)

    task_viz_inputs = attention_tokenizer(
        task_example["input"],
        return_tensors="pt",
        truncation=True,
        max_length=max_viz_length
    )

    with torch.no_grad():
        task_outputs = attention_model(
            **task_viz_inputs,
            output_attentions=True
        )

    task_attn = task_outputs.attentions[layer_idx][0, head_idx].cpu()

    axes[i].imshow(task_attn, aspect="auto")
    axes[i].set_title(f"{task_name} ({task_split}) | Layer {layer_idx} Head {head_idx}")
    axes[i].set_xlabel("Key position")
    axes[i].set_ylabel("Query position")

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


## Wikitext Baseline (Language Modeling)

Wikitext is a language modeling dataset (not sequence to sequence). This cell computes a simple LM baseline on one validation example for both GPT-2 and Pythia-160M using loss/perplexity and generation latency.


In [ ]:
wikitext = load_dataset("wikitext", "wikitext-103-raw-v1", cache_dir=cache_dir)

wiki_text_example = next(
    text for text in wikitext["validation"]["text"]
    if isinstance(text, str) and text.strip()
)


def compute_lm_baseline_on_one_text(model, tokenizer, text, eval_max_length=1024, prompt_tokens=256, max_new_tokens=64):
    model.eval()

    encoded = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=eval_max_length
    )
    input_ids = encoded["input_ids"]
    attention_mask = encoded.get("attention_mask")

    with torch.no_grad():
        lm_out = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=input_ids
        )

    loss = lm_out.loss.item()
    perplexity = torch.exp(torch.tensor(loss)).item()

    prompt_len = min(prompt_tokens, input_ids.shape[1])
    prompt_ids = input_ids[:, :prompt_len]
    prompt_attention_mask = attention_mask[:, :prompt_len] if attention_mask is not None else None

    _ = model.generate(
        input_ids=prompt_ids,
        attention_mask=prompt_attention_mask,
        max_new_tokens=max_new_tokens
    )

    start = time.time()
    generated = model.generate(
        input_ids=prompt_ids,
        attention_mask=prompt_attention_mask,
        max_new_tokens=max_new_tokens
    )
    latency = time.time() - start

    generated_tokens = generated.shape[1] - prompt_ids.shape[1]
    tokens_per_sec = generated_tokens / latency if latency > 0 else float("inf")

    return {
        "loss": loss,
        "perplexity": perplexity,
        "latency_sec": latency,
        "tokens_per_sec": tokens_per_sec,
        "prompt_tokens": prompt_len,
        "generated_tokens": generated_tokens,
    }


wikitext_rows = []
for model_label, model_obj, tok_obj in [
    ("gpt2", gpt2_model, gpt2_tokenizer),
    ("EleutherAI/pythia-160m", pythia_model, pythia_tokenizer),
]:
    metrics = compute_lm_baseline_on_one_text(model_obj, tok_obj, wiki_text_example)
    wikitext_rows.append({"model": model_label, **metrics})

wikitext_df = pd.DataFrame(wikitext_rows)
wikitext_df


## Sliding-Window Local Causal Attention (GPT-2 and Pythia)

This section patches attention modules only (keeps pretrained weights unchanged).


In [ ]:
from transformers.models.gpt2.modeling_gpt2 import GPT2Attention

try:
    from transformers.models.gpt_neox.modeling_gpt_neox import GPTNeoXAttention
    HAS_GPT_NEOX_ATTN = True
except Exception as e:
    GPTNeoXAttention = None
    HAS_GPT_NEOX_ATTN = False
    print(f"GPTNeoXAttention import unavailable in this transformers build: {e}")


def make_sliding_window_causal_mask(seq_len, window_size, device):
    # returns [seq_len, seq_len] boolean mask
    # True = allowed attention
    # Each position i can attend to [max(0, i - window_size + 1), ..., i]
    positions = torch.arange(seq_len, device=device)
    i = positions[:, None]
    j = positions[None, :]
    return (j <= i) & (j >= i - window_size + 1)


def _make_query_key_sliding_mask(query_len, key_len, window_size, device):
    if query_len == key_len:
        return make_sliding_window_causal_mask(query_len, window_size, device)
    full = make_sliding_window_causal_mask(key_len, window_size, device)
    return full[-query_len:, :]


class SlidingWindowGPT2Attention(GPT2Attention):
    def __init__(self, config, is_cross_attention=False, layer_idx=None, window_size=64):
        super().__init__(config, is_cross_attention=is_cross_attention, layer_idx=layer_idx)
        self.window_size = window_size

    def _attn(self, query, key, value, attention_mask=None, head_mask=None, **kwargs):
        scores = torch.matmul(query, key.transpose(-1, -2))

        if getattr(self, "scale_attn_weights", True):
            scores = scores / (value.size(-1) ** 0.5)
        if getattr(self, "scale_attn_by_inverse_layer_idx", False):
            layer_idx = getattr(self, "layer_idx", 0)
            scores = scores / float(layer_idx + 1)

        q_len = query.size(-2)
        k_len = key.size(-2)
        mask_2d = _make_query_key_sliding_mask(q_len, k_len, self.window_size, scores.device)
        mask = mask_2d[None, None, :, :]

        scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)

        if attention_mask is not None:
            scores = scores + attention_mask

        attn_weights = torch.softmax(scores, dim=-1)
        attn_weights = attn_weights.type(value.dtype)
        attn_weights = self.attn_dropout(attn_weights)

        if head_mask is not None:
            attn_weights = attn_weights * head_mask

        attn_output = torch.matmul(attn_weights, value)
        return attn_output, attn_weights


def replace_gpt2_attn_with_sliding_window(model, window_size):
    for layer_idx, block in enumerate(model.transformer.h):
        old_attn = block.attn
        new_attn = SlidingWindowGPT2Attention(
            model.config,
            is_cross_attention=getattr(old_attn, "is_cross_attention", False),
            layer_idx=getattr(old_attn, "layer_idx", layer_idx),
            window_size=window_size,
        )
        new_attn.load_state_dict(old_attn.state_dict(), strict=False)

        ref_param = next(old_attn.parameters(), None)
        if ref_param is not None:
            new_attn.to(device=ref_param.device, dtype=ref_param.dtype)

        block.attn = new_attn

    return model


if HAS_GPT_NEOX_ATTN:
    class SlidingWindowGPTNeoXAttention(GPTNeoXAttention):
        def __init__(self, config, layer_idx=None, window_size=64):
            super().__init__(config, layer_idx=layer_idx)
            self.window_size = window_size

        def _attn(self, query, key, value, attention_mask=None, head_mask=None, **kwargs):
            scores = torch.matmul(query, key.transpose(-1, -2))
            norm_factor = getattr(self, "norm_factor", value.size(-1) ** 0.5)
            scores = scores / norm_factor

            q_len = query.size(-2)
            k_len = key.size(-2)
            mask_2d = _make_query_key_sliding_mask(q_len, k_len, self.window_size, scores.device)
            mask = mask_2d[None, None, :, :]
            scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)

            if attention_mask is not None:
                scores = scores + attention_mask

            attn_weights = torch.softmax(scores, dim=-1)
            attn_weights = attn_weights.type(value.dtype)

            dropout = getattr(self, "attention_dropout", None)
            if dropout is None:
                dropout = getattr(self, "attn_dropout")
            attn_weights = dropout(attn_weights)

            if head_mask is not None:
                attn_weights = attn_weights * head_mask

            attn_output = torch.matmul(attn_weights, value)
            return attn_output, attn_weights


    def replace_pythia_attn_with_sliding_window(model, window_size):
        if not hasattr(model, "gpt_neox"):
            raise ValueError("Expected a GPTNeoX-based model for Pythia patching.")

        for layer_idx, block in enumerate(model.gpt_neox.layers):
            old_attn = block.attention
            new_attn = SlidingWindowGPTNeoXAttention(
                model.config,
                layer_idx=getattr(old_attn, "layer_idx", layer_idx),
                window_size=window_size,
            )
            new_attn.load_state_dict(old_attn.state_dict(), strict=False)

            ref_param = next(old_attn.parameters(), None)
            if ref_param is not None:
                new_attn.to(device=ref_param.device, dtype=ref_param.dtype)

            block.attention = new_attn

        return model


In [ ]:
# Minimal GPT-2 example requested in the deliverable.
window_size = 64

gpt2_sw_tokenizer = AutoTokenizer.from_pretrained("gpt2")
gpt2_sw_model = GPT2LMHeadModel.from_pretrained("gpt2")
gpt2_sw_model = replace_gpt2_attn_with_sliding_window(gpt2_sw_model, window_size)
gpt2_sw_model.eval()

gpt2_inputs = gpt2_sw_tokenizer("Sliding window attention test for GPT-2.", return_tensors="pt")
with torch.no_grad():
    gpt2_outputs = gpt2_sw_model(**gpt2_inputs)

print("GPT-2 sliding-window logits shape:", gpt2_outputs.logits.shape)


In [ ]:
# Matching Pythia test using the same sliding-window idea.
if HAS_GPT_NEOX_ATTN:
    pythia_sw_tokenizer = AutoTokenizer.from_pretrained("EleutherAI/pythia-160m")
    pythia_sw_model = AutoModelForCausalLM.from_pretrained("EleutherAI/pythia-160m")
    pythia_sw_model = replace_pythia_attn_with_sliding_window(pythia_sw_model, window_size)
    pythia_sw_model.eval()

    pythia_inputs = pythia_sw_tokenizer("Sliding window attention test for Pythia.", return_tensors="pt")
    with torch.no_grad():
        pythia_outputs = pythia_sw_model(**pythia_inputs)

    print("Pythia sliding-window logits shape:", pythia_outputs.logits.shape)
else:
    print("Skipped Pythia sliding-window test because GPTNeoXAttention is unavailable in this transformers build.")
